================================================================================
模块 05：量化投资多因子与统计套利 · 第 4/5 课
第 4 课：机器学习多因子组合与非线性特征挖掘 (参考答案版)
================================================================================

【Q1 深度参考答案】
金融时序中的样本重叠与 Purging / Embargo 机制：传统机器学习的 K-Fold 随机交叉验证为什么在金融资产回报率预测中是完全失效的？请从前向持有期重叠（Overlapping Labels）导致的严重信息泄露（Information Leakage）推导 López de Prado 的 Purged K-Fold 算法设计原理。
- **[前向收益标签的时间跨度重叠]**：在量化金融预测中，样本的预测标签通常是未来一段时间的累计收益率（如未来 5 天收益 $y_t = (P_{t+5} - P_t)/P_t$）。如果采用传统随机抽样切分，时刻 $t$ 被划入测试集，而时刻 $t+1$ 被划入训练集。时刻 $t+1$ 的标签包含了 $P_{t+2} \sim P_{t+6}$ 的价格信息，而这些价格直接决定了测试集样本 $t$ 的走势。模型在训练阶段实际上“偷窥”到了测试集持仓期的未来真实价格，导致严重的信息泄露。
- **[Purging（样本重叠清洗）的微观物理]**：López de Prado (2018) 提出必须在时序切分边界施加强制清洗。凡是训练集样本的时间标签区间与测试集的评估时间区间产生任何交叠的观测点，必须从训练集中彻底物理删除（Purged），保证训练集与测试集在事件时间线上互不干涉。
- **[Embargo（隔离期缓冲）防自回归记忆泄露]**：金融资产价格与波动率具有自回归自相关性（AR process）。在测试集结束紧接着进入下一次训练时，测试集末尾的信息会通过自相关残存渗透到训练集中。因此，必须在测试集之后设置一段隔离冷却期（Embargo，通常为样本半衰期的 1~2 倍），彻底斩断微观记忆的泄漏链条。

【Q2 深度参考答案】
线性多因子加权 vs 树模型集成（GBDT/LightGBM）：在挖掘 Alpha 时，树模型相比传统线性 IC 加权的核心优势是什么？树模型如何天然捕获多因子之间的“非线性条件交互”（例如：只有在低估值且高研发支出时，动量因子才显著有效）？
- **[线性模型的固有结构性缺陷]**：线性 IC 加权假定：因子 $A$ 对预期收益的贡献与因子 $B$ 完全无关，且边际效用恒定。然而在真实商业世界中，单看某个因子往往毫无意义（例如：高研发支出在一家高负债、现金流枯竭的破产边缘公司身上是催命符；而在一家零负债、高 ROIC 的科技巨头身上则是巨大的护城河）。
- **[树模型的递归分叉与条件拓扑]**：集成树模型通过贪婪搜索最佳信息增益进行递归分叉。根节点切分“ROIC > 15%”，其左子树进入平庸企业池，右子树进入卓越企业池。在右子树的下一级分裂中，模型可以进一步判定“估值 PE < 25 且 动量 Momentum > 0”。这种层级嵌套决策树天然能够捕获高阶非线性交互（High-order Interactions）与条件阈值，不需要人工手动设计复杂的交叉乘积特征。
- **[对特征单调变换与异常值的非参数鲁棒性]**：树模型的分裂仅取决于特征在横截面上的大小排序（Rank Order），完全不受输入特征极端离群点的干扰，也不需要对特征做严格的正态分布假设，在处理噪声极大的金融数据时具有极高的天然抗噪优势。

【Q3 深度参考答案】
多重假设检验（Multiple Testing）与折损夏普比率（Deflated Sharpe Ratio, DSR）：如果量化研究员尝试了 10,000 个公式化 Alpha 因子，筛选出一个样本内夏普高达 3.0 的因子，为什么投决会必须使用 Bailey & López de Prado 的 DSR 算法对试验次数 $N$ 进行严厉惩罚？伪发现率（FDR）如何摧毁实盘表现？
- **[数据窥探与最大极值统计量的膨胀]**：在统计学极值理论中，即便我们从完全独立的纯高斯随机游走序列中抽取 10,000 组假数据并计算年化夏普比率，根据极值分布定理，这 10,000 个伪策略的最大夏普比率期望值将达到：
  $$\mathbb{E}[\max_{k=1}^N \text{SR}_k] \approx \sqrt{2 \ln N} \approx \sqrt{2 \ln(10000)} \approx 4.29$$
  这意味着，研究员即便完全不具备任何选股能力，单凭无脑尝试 10,000 次，在纯随机世界中也必然能挖掘出样本内夏普大于 3.0 的“假神级因子”。这就是著名的 P-Hacking 与数据挖掘过拟合骗局。
- **[折损夏普比率 DSR 的数学防御]**：Bailey & López de Prado 提出的 DSR 算法直接将“独立试验次数 $N$”以及收益率分布的高阶非正态矩（偏度 $\gamma_3$ 与峰度 $\gamma_4$）代入渐近极值检验。DSR 计算的是：“在经历过 $N$ 次试验筛选后，观测到的夏普比率大于真实夏普的概率”。若未通过 DSR > 0.95 显著性门槛，则证明该因子纯属多重测试的伪发现（False Discovery）。
- **[投决会的制度防线与准入准则]**：投决会必须强制要求量化团队记录每一次失败的回测日志（Backtest Log Tracking），将所有参数网格搜索的组合次数合并计入试验总数 $N$。只有在通过 DSR 严格折损、并且具备清晰商业逻辑闭环的情况下，策略才被批准实盘配置资金。

<div class="callout callout-note">
  <div class="callout-title">📌 投决会执行准则：敬畏过拟合暗礁，坚持样本外严刑拷打</div>
  <div class="callout-body">
    回测曲线越完美，实盘死得越惨烈。真正优秀的量化团队不在于样本内跑出多高的夏普，而在于能否用最冷酷的因果清洗与试验折损算法，将潜伏在数据深处的过拟合幽灵一一击碎。
  </div>
</div>


In [ ]:
from typing import List, Tuple, Dict, Any
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor


def purged_walk_forward_splits(
    n_samples: int,
    train_size: int,
    test_size: int,
    embargo_size: int = 1,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    生成带封锁隔离期 (Embargo) 的前向滚动时间序列交叉验证切分
    """
    splits = []
    start = 0
    while start + train_size + embargo_size + test_size <= n_samples:
        train_idx = np.arange(start, start + train_size)
        test_start = start + train_size + embargo_size
        test_idx = np.arange(test_start, test_start + test_size)
        splits.append((train_idx, test_idx))
        start += test_size
    return splits


def train_and_evaluate_ml_factor_combination(
    features_df: pd.DataFrame,
    target_returns: pd.Series,
    train_size: int = 120,
    test_size: int = 30,
    embargo_size: int = 2,
    n_estimators: int = 30,
    max_depth: int = 3,
) -> Dict[str, Any]:
    """
    基于树模型的滚动 Walk-Forward 因子非线性合成与评价引擎
    """
    splits = purged_walk_forward_splits(
        len(features_df),
        train_size=train_size,
        test_size=test_size,
        embargo_size=embargo_size,
    )

    oos_preds = []
    oos_actuals = []
    feature_importances_list = []

    for tr_idx, te_idx in splits:
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
        )
        X_tr = features_df.iloc[tr_idx]
        y_tr = target_returns.iloc[tr_idx]
        model.fit(X_tr, y_tr)

        X_te = features_df.iloc[te_idx]
        preds = model.predict(X_te)

        oos_preds.extend(preds)
        oos_actuals.extend(target_returns.iloc[te_idx].values)
        feature_importances_list.append(model.feature_importances_)

    oos_preds_s = pd.Series(oos_preds)
    oos_actuals_s = pd.Series(oos_actuals)

    # 计算样本外 Spearman Rank IC 与显著性
    ric, p_val = stats.spearmanr(oos_preds_s, oos_actuals_s)

    # 归一化平均特征重要性
    mean_fi = np.mean(feature_importances_list, axis=0)
    total_fi = np.sum(mean_fi)
    if total_fi > 1e-12:
        mean_fi_norm = mean_fi / total_fi
    else:
        mean_fi_norm = mean_fi
    fi_dict = {col: float(mean_fi_norm[i]) for i, col in enumerate(features_df.columns)}

    # 样本外 Top 20% vs Bottom 20% 多空平均单期超额
    eval_df = pd.DataFrame({"pred": oos_preds_s, "actual": oos_actuals_s})
    q_top = eval_df["pred"].quantile(0.8)
    q_bot = eval_df["pred"].quantile(0.2)
    top_ret = eval_df[eval_df["pred"] >= q_top]["actual"].mean()
    bot_ret = eval_df[eval_df["pred"] <= q_bot]["actual"].mean()
    ls_ret = float(top_ret - bot_ret)

    return {
        "oos_predictions": oos_preds_s,
        "oos_actuals": oos_actuals_s,
        "oos_rank_ic": float(ric),
        "oos_p_value": float(p_val),
        "mean_feature_importances": fi_dict,
        "num_splits": len(splits),
        "oos_long_short_return": ls_ret,
    }


if __name__ == "__main__":
    print("=" * 70)
    print("运行 Lesson 04: 机器学习因子组合与 Walk-Forward 验证单元测试 (参考答案版)")
    print("=" * 70)

    # 1. 验证带 Embargo 的 Walk-Forward 切分正确性
    n_total = 20
    tr_len = 8
    te_len = 3
    emb_len = 2
    splits = purged_walk_forward_splits(n_total, tr_len, te_len, embargo_size=emb_len)

    assert len(splits) > 0, "切分结果列表不能为空"
    for fold_i, (tr_idx, te_idx) in enumerate(splits):
        assert len(tr_idx) == tr_len, f"第 {fold_i} 折训练集长度不符"
        assert len(te_idx) == te_len, f"第 {fold_i} 折测试集长度不符"
        assert max(tr_idx) + emb_len < min(te_idx), f"第 {fold_i} 折未严格满足训练集与测试集间的 Embargo 隔离约束"

    # 2. 构造具有非线性交互特征的多因子面板仿真数据
    np.random.seed(42)
    N = 300
    feature_names = ["VAL", "MOM", "QUAL", "VOL", "GROWTH"]
    n_feats = len(feature_names)

    X_mat = np.random.randn(N, n_feats)
    y_raw = (
        0.35 * X_mat[:, 0] * (X_mat[:, 2] > 0.0)
        + 0.25 * X_mat[:, 1]
        - 0.15 * (X_mat[:, 3] ** 2)
        + 0.40 * np.random.randn(N)
    )

    feat_df = pd.DataFrame(X_mat, columns=feature_names)
    ret_series = pd.Series(y_raw)

    res = train_and_evaluate_ml_factor_combination(
        feat_df,
        ret_series,
        train_size=120,
        test_size=30,
        embargo_size=2,
        n_estimators=30,
        max_depth=3,
    )

    print(f"总计完成 Walk-Forward 滚动折数: {res['num_splits']}")
    print(f"样本外预测点数量: {len(res['oos_predictions'])}")
    print(f"样本外 Spearman Rank IC: {res['oos_rank_ic']:.4f} (p-val: {res['oos_p_value']:.4e})")
    print(f"样本外 Top-Bottom 20% 平均单期多空收益: {res['oos_long_short_return']:.4f}")
    print("各因子平均特征重要性:")
    for f_name, imp in res["mean_feature_importances"].items():
        print(f"  {f_name:8s}: {imp:.4f}")

    # 3. 统计断言检验
    assert res["num_splits"] >= 4, f"切分折数应至少为 4, 实际: {res['num_splits']}"
    assert len(res["oos_predictions"]) == len(res["oos_actuals"]), "样本外预测与真实标签长度必须一致"
    assert res["oos_rank_ic"] > 0.30, f"非线性合成因子样本外 Rank IC 应显著大于 0.30, 实际: {res['oos_rank_ic']:.4f}"
    assert res["oos_long_short_return"] > 0.10, f"样本外多空组合收益应大于 0.10, 实际: {res['oos_long_short_return']:.4f}"
    assert abs(sum(res["mean_feature_importances"].values()) - 1.0) < 1e-4, "特征重要性总和应归一化为 1.0"
    assert (
        res["mean_feature_importances"]["VAL"] > 0.10
        and res["mean_feature_importances"]["MOM"] > 0.10
    ), "核心特征 VAL 或 MOM 重要性未能被模型有效识别"

    print("Lesson 04 参考答案单元测试全部通过 (Exit Code 0)！")
